# 03 — Autoformer with Uncertainty Quantification
This notebook introduces the core contribution: an Autoformer utilizing Series Decomposition and FFT-based Auto-Correlation.
The architecture features a dual-headed output optimized via Gaussian Negative Log-Likelihood to predict both the expected return and dynamic uncertainty boundaries.

In [1]:
import os, sys, numpy as np, pandas as pd, torch, torch.nn as nn, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.utils.data import TensorDataset, DataLoader
sys.path.insert(0, os.path.abspath('..'))
DATA_PATH='../data'; SEQ_LEN=60; FORECAST=5; NUM_STOCKS=50; TRAIN_R=0.8; TGT=1; BATCH=64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Ready | device:', device)


Ready | device: cpu


## Data

In [2]:
def make_seq(data):
    X,y=[],[]
    for i in range(len(data)-SEQ_LEN-FORECAST+1):
        X.append(data[i:i+SEQ_LEN])
        y.append(data[i+SEQ_LEN:i+SEQ_LEN+FORECAST, TGT])
    return np.array(X), np.array(y)

def load_data():
    files=[f for f in sorted(os.listdir(DATA_PATH)) if f.endswith('.txt')][:NUM_STOCKS]
    dfs=[]
    for f in files:
        try:
            df=pd.read_csv(os.path.join(DATA_PATH,f))
            if not {'Date','Close'}.issubset(df.columns): continue
            df['Date']=pd.to_datetime(df['Date']); df=df.sort_values('Date')
            df['Return']=df['Close'].pct_change(); df['MA_10']=df['Close'].rolling(10).mean()
            df['MA_50']=df['Close'].rolling(50).mean(); df=df.dropna(); df['Stock']=f
            if len(df)>SEQ_LEN+FORECAST: dfs.append(df)
        except: pass
    return pd.concat(dfs,ignore_index=True)

cdf=load_data()
all_X,all_y=[],[]
for s in cdf['Stock'].unique():
    sdf=cdf[cdf['Stock']==s]; sc=MinMaxScaler()
    d=sc.fit_transform(sdf[['Close','Return','MA_10','MA_50']]); Xs,ys=make_seq(d)
    if len(Xs): all_X.append(Xs); all_y.append(ys)
X=np.concatenate(all_X); y=np.concatenate(all_y)
sp=int(TRAIN_R*len(X)); X_tr,X_te=X[:sp],X[sp:]; y_tr,y_te=y[:sp],y[sp:]
T=lambda a: torch.tensor(a,dtype=torch.float32).to(device)
Xt,Xe,yt_t,ye_t=T(X_tr),T(X_te),T(y_tr),T(y_te)
print(f'Train:{len(X_tr):,} | Test:{len(X_te):,} | Shape X:{X.shape}')


Train:108,644 | Test:27,161 | Shape X:(135805, 60, 4)


## Model Architecture

In [3]:
from src.autoformer_model import StockAutoformer, gnll
print('Autoformer model defined via src import')


Autoformer model defined via src import


## Training (Gaussian NLL Loss)

In [4]:
MODEL_NAME='Autoformer+UQ'; EPOCHS_P1=20; EPOCHS_P2=5; LR=3e-4
model=StockAutoformer(input_size=4).to(device)
print(f'Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

print('\nPhase 1: MSE Training (Accurate Point Predictions)')
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS_P1)
ldr=DataLoader(TensorDataset(Xt,yt_t),batch_size=BATCH,shuffle=True)
losses=[]
for ep in range(EPOCHS_P1):
    model.train(); tot=0
    for xb,yb in ldr:
        m,lv=model(xb); loss=nn.MSELoss()(m,yb)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step(); tot+=loss.item()
    avg=tot/len(ldr); losses.append(avg); sch.step()
    print(f'Epoch {ep+1:02d}/{EPOCHS_P1} MSE:{avg:.6f}')

print('\nPhase 2: GNLL Fine-Tuning (Calibrated Uncertainty)')
opt2=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
for ep in range(EPOCHS_P2):
    model.train(); tot=0
    for xb,yb in ldr:
        m,lv=model(xb); loss=gnll(m,lv,yb)
        opt2.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt2.step(); tot+=loss.item()
    avg=tot/len(ldr); losses.append(avg)
    print(f'Epoch {ep+1:02d}/{EPOCHS_P2} GNLL:{avg:.6f}')

torch.save(model.state_dict(),'../models/autoformer_uq_model.pth')
model.eval()
with torch.no_grad():
    mp,lv=model(Xe); mu=mp.cpu().numpy()
    std=torch.exp(0.5*lv).cpu().numpy()


Params: 467,854

Phase 1: MSE Training (Accurate Point Predictions)


Epoch 01/20 MSE:0.009511


Epoch 02/20 MSE:0.004606


Epoch 03/20 MSE:0.004178


Epoch 04/20 MSE:0.003990


...

Epoch 01/5 GNLL:-2.088978


Epoch 02/5 GNLL:-2.367639


Epoch 03/5 GNLL:-2.424377


Epoch 04/5 GNLL:-2.450777


Epoch 05/5 GNLL:-2.466742


## Save Results

In [5]:
os.makedirs('../outputs',exist_ok=True)
rows=[]
for d in range(FORECAST):
    mae=mean_absolute_error(y_te[:,d],mu[:,d]); rmse=float(np.sqrt(mean_squared_error(y_te[:,d],mu[:,d])))
    avg_std=float(std[:,d].mean())
    rows.append({'model':MODEL_NAME,'day':f'Day+{d+1}','mae':round(mae,6),'rmse':round(rmse,6),'avg_uncertainty':round(avg_std,6)})
ov_mae=mean_absolute_error(y_te.flatten(),mu.flatten()); ov_rmse=float(np.sqrt(mean_squared_error(y_te.flatten(),mu.flatten())))
rows.append({'model':MODEL_NAME,'day':'Overall','mae':round(ov_mae,6),'rmse':round(ov_rmse,6),'avg_uncertainty':None})
rp='../outputs/results.csv'; new=pd.DataFrame(rows)
if os.path.exists(rp):
    ex=pd.read_csv(rp); ex=ex[ex['model']!=MODEL_NAME]; new=pd.concat([ex,new],ignore_index=True)
new.to_csv(rp,index=False); print('Results saved'); print(new[new['model']==MODEL_NAME].to_string(index=False))


Results saved
        model     day      mae     rmse  avg_uncertainty
Autoformer+UQ   Day+1 0.056874 0.074026         0.087965
Autoformer+UQ   Day+2 0.053402 0.071256         0.083663
Autoformer+UQ   Day+3 0.059074 0.076146         0.088425
Autoformer+UQ   Day+4 0.057790 0.074921         0.090260
Autoformer+UQ   Day+5 0.056516 0.073837         0.087057
Autoformer+UQ Overall 0.056731 0.074055              NaN


## Plots

In [6]:
plt.figure(figsize=(8,4)); plt.plot(losses,color='#E67E22',lw=2)
plt.title('Autoformer+UQ Training Loss (GNLL)'); plt.xlabel('Epoch'); plt.ylabel('GNLL')
plt.tight_layout(); plt.savefig('../outputs/autoformer_uq_loss.png',dpi=150); plt.close()

n=min(300,len(y_te)); idx=np.arange(n)
plt.figure(figsize=(14,5))
plt.plot(idx,y_te[:n,0],label='Actual',color='#2ECC71',lw=1.2)
plt.plot(idx,mu[:n,0],label='Mean Prediction',color='#E74C3C',lw=1.2,alpha=0.9)
plt.fill_between(idx,mu[:n,0]-std[:n,0],mu[:n,0]+std[:n,0],alpha=0.3,color='#E74C3C',label='+/- 1 Std')
plt.fill_between(idx,mu[:n,0]-2*std[:n,0],mu[:n,0]+2*std[:n,0],alpha=0.1,color='#E74C3C',label='+/- 2 Std')
plt.title('Autoformer+UQ: Predictions with Confidence Bands (Day+1)'); plt.xlabel('Sample'); plt.ylabel('Scaled Return')
plt.legend(); plt.tight_layout(); plt.savefig('../outputs/autoformer_uq_bands.png',dpi=150); plt.close()

plt.figure(figsize=(6,4))
plt.bar([f'Day+{i+1}' for i in range(FORECAST)],std.mean(0),color='#E67E22')
plt.title('Average Uncertainty per Forecast Day'); plt.ylabel('Mean Uncertainty (Std)')
plt.tight_layout(); plt.savefig('../outputs/autoformer_uq_uncertainty.png',dpi=150); plt.close()
print('Plots saved')


Plots saved


## Conclusion
- Autoformer isolates market trends via Series Decomposition.
- FFT Auto-Correlation captures periodic dependencies more effectively than dot-product attention.
- The dual-headed output provides calibrated risk assessment alongside predictions.
- Reference: Wu et al., NeurIPS 2021 — *Autoformer: Decomposition Transformers with Auto-Correlation*